<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Assignment_1_Python_Mastery_%26_LLM_API_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =======================================================
# Assignment 1: Python Mastery & LLM API Integration
# This notebook demonstrates programmatic interaction with an LLM API.
# It includes essential error handling and retry logic.
# =======================================================

# You may need to install the 'requests' library if it's not already
# installed in your Colab environment.
# Uncomment the line below and run this cell once if you encounter an error.
# !pip install requests

import requests
import json
import time

# --- Setup for API Call ---
# IMPORTANT: Replace 'YOUR_API_KEY_HERE' with your actual API key.
# In a real application, you would load this from an environment variable.
API_KEY = "YOUR_API_KEY_HERE"
API_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent"

# A sample prompt for our project: generating a code-switched dialogue
# This is where you would place the advanced prompts from your Week 2 lab!
prompt_text = """
Act as a bilingual friend fluent in English and Spanish.
Write a short dialogue where you discuss a favorite restaurant.
The conversation should include at least one instance of code-switching
where a food item is mentioned in Spanish.
"""

def generate_content_with_retry(api_key, api_url, prompt, max_retries=5):
    """
    Makes an API call to a generative model with retry logic.
    Implements a simple exponential backoff strategy for retries.
    """
    headers = {
        "Content-Type": "application/json",
    }

    # We define the payload that includes our prompt and generation settings
    payload = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ],
        "generationConfig": {
            "temperature": 0.7,  # A good balance between creativity and consistency
            "maxOutputTokens": 200 # Set a limit to prevent overly long responses
        }
    }

    # Loop for retry logic
    retries = 0
    while retries < max_retries:
        try:
            print(f"Attempting API call (Attempt {retries + 1})...")
            # The API key is added as a query parameter in the URL
            response = requests.post(f"{api_url}?key={api_key}", headers=headers, data=json.dumps(payload))

            # Raise an HTTPError if the status code is an error (4xx or 5xx)
            response.raise_for_status()

            # If the request is successful, parse the JSON response
            result = response.json()

            # The generated text is typically nested in the response JSON
            generated_text = result['candidates'][0]['content']['parts'][0]['text']
            return generated_text

        except requests.exceptions.RequestException as e:
            # This block handles any type of request error (network, HTTP errors, etc.)
            print(f"API call failed: {e}")

            if retries < max_retries - 1:
                wait_time = 2 ** retries  # Exponential backoff: 1s, 2s, 4s, etc.
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
                retries += 1
            else:
                print("Max retries reached. API call failed permanently.")
                return None

    return None

# --- Main Execution ---
if __name__ == "__main__":
    if API_KEY == "YOUR_API_KEY_HERE":
        print("Please replace 'YOUR_API_KEY_HERE' with your actual API key.")
        print("This script will not run until the key is provided.")
    else:
        # Call the function to get the generated text
        dialogue = generate_content_with_retry(API_KEY, API_URL, prompt_text)

        if dialogue:
            print("\n--- Generated Dialogue ---")
            print(dialogue)
        else:
            print("\nCould not generate dialogue after multiple attempts.")